In [ ]:
# Лабораторная работа 4

In [ ]:
import pandas as pd
import numpy as np

order_items = pd.read_csv("/content/olist_order_items_dataset.csv")
orders = pd.read_csv('/content/olist_orders_dataset.csv')

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Задание 1: Бизнес-календарь и нарушения SLA

In [ ]:
'''Типизация и индексы: Преобразуйте даты в datetime64.
Создайте копию датасета, где индексом будет выступать
order_purchase_timestamp.'''

date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

orders[date_columns] = orders[date_columns].astype('datetime64[ns]')

data = orders.set_index('order_purchase_timestamp').copy()
data

,order_id,customer_id,order_status,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_purchase_timestamp,,,,,,,
2017-10-02 10:56:33,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
2018-07-24 20:41:37,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2018-08-08 08:38:49,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
2017-11-18 19:28:06,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
2018-02-13 21:18:39,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26
...,...,...,...,...,...,...,...
2017-03-09 09:54:05,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28
2018-02-06 12:58:58,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02
2017-08-27 14:46:43,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27


In [ ]:
'''Рабочие дни (Business Days): Рассчитайте
delivery_time_working_days — количество исключительно рабочих дней
между подтверждением заказа (order_approved_at) и его доставкой
(order_delivered_customer_date).
Подсказка: изучите np.busday_count или pd.bdate_range.'''

mask = (data['order_approved_at'].notna() &
        data['order_delivered_customer_date'].notna()
        )
data['delivery_time_working_days'] = np.nan

data.loc[mask, 'delivery_time_working_days'] = np.busday_count(
    data.loc[mask, 'order_approved_at'].values.astype('datetime64[D]'),
    data.loc[mask, 'order_delivered_customer_date'].values.astype('datetime64[D]')
)
data.to_csv(
    '/content/drive/MyDrive/delivery_data.csv',
    index=False
)
data['delivery_time_working_days']

,delivery_time_working_days
order_purchase_timestamp,
2017-10-02 10:56:33,6.0
2018-07-24 20:41:37,8.0
2018-08-08 08:38:49,7.0
2017-11-18 19:28:06,10.0
2018-02-13 21:18:39,3.0
...,...
2017-03-09 09:54:05,6.0
2018-02-06 12:58:58,16.0
2017-08-27 14:46:43,18.0


In [ ]:
'''Анализ просрочек (SLA Breach): Сравните фактическую дату
доставки с обещанной ( order_estimated_delivery_date).
Создайте булеву маску is_delayed (доставлено позже срока).
Найдите месяц с максимальным процентом просроченных заказов.'''

mask = (data['order_approved_at'].notna() &
        data['order_estimated_delivery_date'].notna()
        )
data['estimated_delivery_time_working_days'] = np.nan

data.loc[mask, 'estimated_delivery_time_working_days'] = np.busday_count(
    data.loc[mask, 'order_approved_at'].values.astype('datetime64[D]'),
    data.loc[mask, 'order_estimated_delivery_date'].values.astype('datetime64[D]')
)
data['is_delayed'] = (data['estimated_delivery_time_working_days']
                      < data['delivery_time_working_days']).fillna(False)

data['order_month'] = data['order_approved_at'].dt.to_period('M')

delay_stats = data.groupby('order_month').agg(
    total_orders=('order_id', 'count'),
    delayed_orders=('is_delayed', 'sum')
)

delay_stats['delay_rate'] = (
    delay_stats['delayed_orders'] / delay_stats['total_orders'] * 100)

print("Mесяц с максимальным процентом просроченных заказов:",
      delay_stats['delay_rate'].idxmax())

data.to_csv(
    '/content/drive/MyDrive/review_data.csv',
    index=False
)
data

Mесяц с максимальным процентом просроченных заказов: 2016-09


,order_id,customer_id,order_status,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_working_days,estimated_delivery_time_working_days,is_delayed,order_month
order_purchase_timestamp,,,,,,,,,,,
2017-10-02 10:56:33,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,6.0,12.0,False,2017-10
2018-07-24 20:41:37,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,8.0,12.0,False,2018-07
2018-08-08 08:38:49,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,7.0,19.0,False,2018-08
2017-11-18 19:28:06,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,10.0,19.0,False,2017-11
2018-02-13 21:18:39,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,3.0,9.0,False,2018-02
...,...,...,...,...,...,...,...,...,...,...,...
2017-03-09 09:54:05,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,6.0,13.0,False,2017-03
2018-02-06 12:58:58,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,16.0,18.0,False,2018-02
2017-08-27 14:46:43,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,18.0,22.0,False,2017-08


In [ ]:
'''Непрерывная сетка: Посчитайте суммарную выручку по дням.
Найдите минимальную и максимальную даты в датасете.
Постройте полную сетку дат ( pd.date_range) и выполните
reindex вашего сгруппированного ряда.
Заполните образовавшиеся NaN (дни без продаж) нулями.'''

order_items['shipping_limit_date'] = pd.to_datetime(
    order_items['shipping_limit_date']
)

order_items['day_date'] = order_items['shipping_limit_date'].dt.to_period('D')
revenue = order_items.groupby('day_date')['price'].sum()

max_date = revenue.index.max()
min_date = revenue.index.min()
print(max_date, min_date)

dates = pd.date_range(min_date.to_timestamp(), max_date.to_timestamp())
revenue.index = revenue.index.to_timestamp()
revenue = revenue.reindex(dates, fill_value=0)

revenue1 = revenue.reset_index()

revenue1.columns = ['index', 'price']
revenue1['index'] = pd.to_datetime(revenue1['index'])

revenue1.to_csv(
    '/content/drive/MyDrive/revenue.csv',
    index=False
)
revenue1


2020-04-09 2016-09-19


,index,price
0,2016-09-19,194.47
1,2016-09-20,0.00
2,2016-09-21,0.00
3,2016-09-22,0.00
4,2016-09-23,0.00
...,...,...
1294,2020-04-05,0.00
1295,2020-04-06,0.00
1296,2020-04-07,0.00
1297,2020-04-08,0.00


In [ ]:
'''Пиковые часы со сдвигом: Обычное среднее количество заказов
по часам непоказательно (в выходные люди покупают в другое время).
С помощью сводной таблицы ( pivot_table) или двойной группировки
найдите час пик продаж для каждого конкретного дня недели'''

order_items['day_hours'] = order_items['shipping_limit_date'].dt.hour
order_items['week_day'] = order_items['shipping_limit_date'].dt.day_of_week

pivot = pd.pivot_table(order_items,
               'order_id',
               'week_day',
               'day_hours',
               'count',
               fill_value=0
               )
print(pivot.idxmax(axis=1))
pivot.to_csv(
    '/content/drive/MyDrive/day_hour.csv',
    index=False
)
pivot

week_day
0     3
1     2
2     2
3     2
4    16
5    13
6    22
dtype: int32


day_hours,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
week_day,,,,,,,,,,,,,,,,,,,,,
0,390,240,642,2073,1564,446,255,419,477,890,...,1030,1084,972,1037,912,917,892,937,985,693
1,459,282,1803,1237,698,146,169,292,544,928,...,1044,1177,1019,1064,929,925,980,903,893,608
2,399,247,1771,1152,604,120,183,251,471,758,...,1002,994,1135,1019,865,866,888,1361,1530,1152
3,712,613,3175,1232,874,240,326,391,620,1210,...,1719,1713,1791,1705,1496,1603,1752,1112,1010,763
4,428,257,1010,674,327,99,133,240,498,776,...,1268,1113,1411,1282,1031,1065,1042,323,21,22
5,8,3,2,1,10,10,1,5,12,26,...,35,25,20,30,21,32,18,27,28,24
6,7,7,4,8,5,0,6,2,15,17,...,19,29,18,30,28,24,27,731,975,701


# Задание 3: Динамические бейзлайны и локальные аномалии


In [ ]:
'''Кумулятивные метрики (YTD):
Рассчитайте кумулятивную (накопительную) выручку с начала года
(Year-To-Date) для каждого дня, используя метод .expanding().'''

revenue.index = pd.to_datetime(revenue.index)
revenue = revenue.to_frame(name='daily_revenue')
revenue['year'] = revenue.index.year

revenue['revenue_ytd'] = revenue.groupby('year')['daily_revenue'].expanding().sum().reset_index(level=0, drop=True)

revenue


,daily_revenue,year,revenue_ytd
2016-09-19,194.47,2016,194.47
2016-09-20,0.00,2016,194.47
2016-09-21,0.00,2016,194.47
2016-09-22,0.00,2016,194.47
2016-09-23,0.00,2016,194.47
...,...,...,...
2020-04-05,0.00,2020,145.98
2020-04-06,0.00,2020,145.98
2020-04-07,0.00,2020,145.98
2020-04-08,0.00,2020,145.98


In [ ]:
'''Скользящий Z-score: Для поиска аномалий реализуйте динамический порог.
Вычислите 30-дневное скользящее среднее (μt−30) и
30-дневное скользящее стандартное отклонение (σt−30).
Рассчитайте скользящий Z-score по формуле.
Найдите даты, где Zt>3 (локальные аномальные всплески).'''


roll_mean = revenue.rolling(window=30).mean()
roll_std = revenue.rolling(window=30).std()

z = (revenue - roll_mean) / roll_std
a = z[z > 2.5]

revenue = revenue.reset_index()

revenue['index'] = pd.to_datetime(revenue['index'])
anomalies = a['daily_revenue']

anomalies = anomalies.reset_index()

anomalies['index'] = pd.to_datetime(anomalies['index'])

anomalies = pd.merge(anomalies, revenue, on='index', how='left')


anomalies.to_csv(
    '/content/drive/MyDrive/anomalies.csv',
    index=False
)

roll_mean


,daily_revenue,year,revenue_ytd
2016-09-19,NaN,NaN,NaN
2016-09-20,NaN,NaN,NaN
2016-09-21,NaN,NaN,NaN
2016-09-22,NaN,NaN,NaN
2016-09-23,NaN,NaN,NaN
...,...,...,...
2020-04-05,0.000,2020.0,145.980
2020-04-06,0.000,2020.0,145.980
2020-04-07,0.000,2020.0,145.980
2020-04-08,0.000,2020.0,145.980


In [ ]:
roll_mean

,daily_revenue,year,revenue_ytd
2016-09-19,NaN,NaN,NaN
2016-09-20,NaN,NaN,NaN
2016-09-21,NaN,NaN,NaN
2016-09-22,NaN,NaN,NaN
2016-09-23,NaN,NaN,NaN
...,...,...,...
2020-04-05,0.000,2020.0,145.980
2020-04-06,0.000,2020.0,145.980
2020-04-07,0.000,2020.0,145.980
2020-04-08,0.000,2020.0,145.980


In [ ]:
rm = roll_mean.reset_index()

rm.columns = ['index', 'daily_revenue', 'year', 'revenue_ytd']
rm['index'] = pd.to_datetime(rm['index'])

rm.to_csv(
    '/content/drive/MyDrive/roll_mean.csv',
    index=False
)
rm

,index,daily_revenue,year,revenue_ytd
0,2016-09-19,NaN,NaN,NaN
1,2016-09-20,NaN,NaN,NaN
2,2016-09-21,NaN,NaN,NaN
3,2016-09-22,NaN,NaN,NaN
4,2016-09-23,NaN,NaN,NaN
...,...,...,...,...
1294,2020-04-05,0.000,2020.0,145.980
1295,2020-04-06,0.000,2020.0,145.980
1296,2020-04-07,0.000,2020.0,145.980
1297,2020-04-08,0.000,2020.0,145.980


# Задание 4: Лаги и защита от Data Leakage (ML Prep)


In [ ]:
'''Безопасные окна: Рассчитайте признак revenue_last_7d
(выручка за предыдущие 7 дней). Важное условие: данные
за текущий день не должны входить в эту сумму.
Скомбинируйте методы .shift() и .rolling().'''

revenue['revenue_last_7d'] = revenue['daily_revenue'].shift(1).rolling(window=7).sum()
revenue

,index,daily_revenue,year,revenue_ytd,revenue_last_7d
0,2016-09-19,194.47,2016,194.47,NaN
1,2016-09-20,0.00,2016,194.47,NaN
2,2016-09-21,0.00,2016,194.47,NaN
3,2016-09-22,0.00,2016,194.47,NaN
4,2016-09-23,0.00,2016,194.47,NaN
...,...,...,...,...,...
1294,2020-04-05,0.00,2020,145.98,0.0
1295,2020-04-06,0.00,2020,145.98,0.0
1296,2020-04-07,0.00,2020,145.98,0.0
1297,2020-04-08,0.00,2020,145.98,0.0


In [ ]:
'''Относительный темп (WoW Growth): Рассчитайте Week-over-Week рост выручки.
Сравните сумму выручки за последние 7 дней ( revenue_last_7d) с
суммой выручки за аналогичный период неделей ранее ( revenue_prev_7d).
Выведите топ-3 дня с самым сильным падением выручки неделя к неделе.'''

revenue['revenue_prev_7d'] = revenue['daily_revenue'].shift(7).rolling(window=7).sum()
revenue['diff'] = revenue['revenue_prev_7d'] - revenue['revenue_last_7d']
revenue['pct'] = np.where(
    revenue['revenue_prev_7d'] != 0,
    (revenue['diff'] / revenue['revenue_prev_7d']),
    np.nan
)

#Если смотреть на разность
print(revenue.nsmallest(3, 'diff'))

#Если смотреть на долю
revenue.nsmallest(20, 'pct')

         index  daily_revenue  year  revenue_ytd  revenue_last_7d  \
635 2018-06-16           0.00  2018   5190724.29        321166.30   
636 2018-06-17       12788.49  2018   5203512.78        321166.30   
442 2017-12-05       50708.93  2017   5308931.71        390433.22   

     revenue_prev_7d       diff       pct  
635        122229.82 -198936.48 -1.627561  
636        130714.98 -190451.32 -1.456997  
442        208909.06 -181524.16 -0.868915  


,index,daily_revenue,year,revenue_ytd,revenue_last_7d,revenue_prev_7d,diff,pct
26,2016-10-15,2400.16,2016,32752.85,30158.22,908.29,-29249.93,-32.203294
119,2017-01-16,1028.89,2017,8369.59,7340.70,396.90,-6943.80,-17.495087
27,2016-10-16,1658.59,2016,34411.44,31650.09,2453.86,-29196.23,-11.898083
120,2017-01-17,2435.68,2017,10805.27,7972.69,1203.38,-6769.31,-5.625247
121,2017-01-18,842.58,2017,11647.85,9601.89,2185.28,-7416.61,-3.393895
122,2017-01-19,2400.37,2017,14048.22,9462.57,2505.07,-6957.50,-2.777367
123,2017-01-20,3343.70,2017,17391.92,11543.15,3308.85,-8234.30,-2.488569
28,2016-10-17,561.09,2016,34972.53,31763.11,10061.77,-21701.34,-2.156811
124,2017-01-21,2971.73,2017,20363.65,14083.07,5308.62,-8774.45,-1.652868
635,2018-06-16,0.00,2018,5190724.29,321166.30,122229.82,-198936.48,-1.627561
